In [1]:
import asyncio
import httpx
import logging
import json
import sys
import os
from pathlib import Path
import time

In [6]:
import json

In [2]:
from typing import Dict, Any, List, Optional, Tuple, Union

In [31]:
import requests

In [26]:
from pydantic import BaseModel
from typing import List, Dict, Union, Optional, Any

class CurrentState(BaseModel):
    evaluation_previous_goal: str  # "成功|失败|未知 - 对前一步操作结果的分析"
    memory: str  # "已完成步骤的描述和需要记住的上下文信息"
    next_goal: str  # "下一步操作的目标"
    user_interaction_needed: bool  # 标记是否需要用户交互

class Action(BaseModel):
    action_name: str
    parameters: Dict[str, Any]

class RequestUserAction(BaseModel):
    type: str  # "login|select|verify|input|decision"
    message: str  # "请用户执行的操作描述"
    description: str  # "详细说明"
    options: Optional[List[str]] = None  # 可选参数，提供选择项

class EvaluateState(BaseModel):
    description: str  # "用户完成操作的描述"

class WorkflowStep(BaseModel):
    current_state: CurrentState
    action: List[Union[Action, RequestUserAction, EvaluateState]]



In [21]:
import json
from pydantic import BaseModel
from typing import Type

def get_format_instructions(pydantic_object: Type[BaseModel]) -> str:
    """Return the format instructions for the JSON output based on a Pydantic model.
    
    Args:
        pydantic_object: The Pydantic model class (not an instance) to generate format for
        
    Returns:
        The format instructions for the JSON output that can be used in an LLM prompt
    """
    # Get the JSON schema for the Pydantic model
    schema = pydantic_object.model_json_schema()
    
    # Remove extraneous fields to simplify the schema
    reduced_schema = schema.copy()
    if "title" in reduced_schema:
        del reduced_schema["title"]
    if "type" in reduced_schema:
        del reduced_schema["type"]
    
    # Create example based on the schema structure
    example = create_example_from_schema(schema)
    example_str = json.dumps(example, ensure_ascii=False, indent=2)
    
    # Ensure schema is well-formed with double quotes
    schema_str = json.dumps(reduced_schema, ensure_ascii=False, indent=2)
    
    return PYDANTIC_FORMAT_INSTRUCTIONS.format(
        schema=schema_str,
        example=example_str,
        model_name=pydantic_object.__name__
    )
# 先定义模板字符串
_PYDANTIC_FORMAT_INSTRUCTIONS = """The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {{"properties": {{"foo": {{"title": "Foo", "description": "a list of strings", "type": "array", "items": {{"type": "string"}}}}}}, "required": ["foo"]}}
the object {{"foo": ["bar", "baz"]}} is a well-formatted instance of the schema. The object {{"properties": {{"foo": ["bar", "baz"]}}}} is not well-formatted.

Here is the output schema:
```
{schema}
```""" 

In [27]:
json_str=get_format_instructions(WorkflowStep)

In [29]:
print(json_str)

# Output Format Instructions

I need you to generate data that conforms to the WorkflowStep model defined below.

## Schema Definition
The output should be formatted as a JSON instance that conforms to this JSON schema:
```json
{
  "$defs": {
    "Action": {
      "properties": {
        "action_name": {
          "title": "Action Name",
          "type": "string"
        },
        "parameters": {
          "additionalProperties": true,
          "title": "Parameters",
          "type": "object"
        }
      },
      "required": [
        "action_name",
        "parameters"
      ],
      "title": "Action",
      "type": "object"
    },
    "CurrentState": {
      "properties": {
        "evaluation_previous_goal": {
          "title": "Evaluation Previous Goal",
          "type": "string"
        },
        "memory": {
          "title": "Memory",
          "type": "string"
        },
        "next_goal": {
          "title": "Next Goal",
          "type": "string"
        },
    

In [71]:
from autogen_core.code_executor import CodeBlock,CodeExecutor
from autogen_ext.code_executors.docker import DockerCommandLineCodeExecutor
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor
from autogen_ext.code_executors.jupyter import JupyterCodeExecutor
import tempfile
from pathlib import Path
import venv
import asyncio
from autogen_core import CancellationToken
async def LocalCodeExecutor(codeblock_list,env=None,filedir='/oper/ch/autogen'):
    work_dir = Path(filedir)
    work_dir.mkdir(exist_ok=True)
    if not env:
        venv_dir = work_dir / ".venv"
        venv_builder = venv.EnvBuilder(with_pip=True)
        venv_builder.create(venv_dir)
        venv_context = venv_builder.ensure_directories(venv_dir)
    else:
        venv_builder = venv.EnvBuilder(with_pip=True)
        venv_context = venv_builder.ensure_directories(env)
    local_executor = LocalCommandLineCodeExecutor(work_dir=work_dir, virtual_env_context=venv_context)
    try:
        result = await local_executor.execute_code_blocks(
            code_blocks=codeblock_list,
            cancellation_token=CancellationToken(),)
        return result.output
    except Exception as e:
        return f"错误问题: {e}"

In [4]:
from browser_use.browser.browser import Browser, BrowserConfig
from browser_use.browser.context import BrowserContext, BrowserContextConfig

INFO     [browser_use] BrowserUse logging setup complete with level info
INFO     [root] Anonymized telemetry enabled. See https://docs.browser-use.com/development/telemetry for more information.


In [5]:
chrome_debug_port=54905
browser_config = BrowserConfig(
    cdp_url=f"http://localhost:{chrome_debug_port}"
)

# 创建Browser实例
browser = Browser(config=browser_config)

# 创建BrowserContext
browser_context = BrowserContext(browser=browser)

browser_context._initialize_session()

<coroutine object BrowserContext._initialize_session at 0x7cf803efc6a0>

In [5]:
await browser_context.create_new_tab(url='https://www.google.com')

In [9]:
current_path="/oper/work/endian/intelligent_agent"
sys.path.append(current_path)

In [23]:
from tool_service.src.tools.handlers.base import BaseHandler
base_handler = BaseHandler(browser_context)

In [20]:
parameters= {
    "base_url": "https://www.gdzwfw.gov.cn/portal/index?region=440300",
    "search_keyword": "社保清单打印",
    "search_button_text": "搜索",
    "result_keyword": "社保查询",
    "wait_after_search": 2
}
await base_handler.search_and_navigate(parameters)

INFO     [tool_service.src.tools.handlers.base] 未找到包含URL'https://www.gdzwfw.gov.cn/portal/index?region=440300'的标签页，创建新标签页...


{'status': 'success',
 'message': "成功搜索并导航到'社保查询'相关页面",
 'new_tab_created': True,
 'url': 'https://www.gdzwfw.gov.cn/portal/v3/lv3/hot?region=440300&area_code=440300&type=person&id=7900000468&lv1code=7700000011&lv1_name=%E7%A4%BE%E4%BF%9D&lv2_name=%E7%A4%BE%E4%BF%9D%E6%9F%A5%E8%AF%A2'}

In [24]:
parameters= {
    "base_url": "https://www.google.com",
    "search_keyword": "tiktok global shop",
    "search_button_text": "google search",
    "result_keyword": "tiktok",
    "wait_after_search": 2
}
await base_handler.search_and_navigate(parameters)

INFO     [tool_service.src.tools.handlers.base] 找到包含URL'https://www.google.com'的标签页，ID为: 4


{'status': 'error', 'message': '未找到搜索输入框'}

In [18]:
selector_map = await browser_context.get_selector_map()
                        
# 尝试匹配找到的input元素
for idx, element in selector_map.items():
    print(idx,element)
    if element.tag_name.lower() == "input":
        search_box_index = idx
        break

0 <a class="MV3Tnb" href="https://about.google/?fg=1&utm_source=google-SG&utm_medium=referral&utm_campaign=hp-header" ping="/url?sa=t&rct=j&source=webhp&url=https://about.google/%3Ffg%3D1%26utm_source%3Dgoogle-SG%26utm_medium%3Dreferral%26utm_campaign%3Dhp-header&ved=0ahUKEwiVuZzA-diMAxVLyzgGHdhZE_oQkNQCCAI&opi=89978449"> [interactive, top, highlight:0, in-viewport]
1 <a class="MV3Tnb" href="https://store.google.com/SG?utm_source=hp_header&utm_medium=google_ooo&utm_campaign=GS100042&hl=en-SG" ping="/url?sa=t&rct=j&source=webhp&url=https://store.google.com/SG%3Futm_source%3Dhp_header%26utm_medium%3Dgoogle_ooo%26utm_campaign%3DGS100042%26hl%3Den-SG&ved=0ahUKEwiVuZzA-diMAxVLyzgGHdhZE_oQpMwCCAM&opi=89978449"> [interactive, top, highlight:1, in-viewport]
2 <a class="gb_X" aria-label="Gmail " data-pid="23" href="https://mail.google.com/mail/&ogbl" target="_top"> [interactive, top, highlight:2, in-viewport]
3 <a class="gb_X" aria-label="Search for Images " data-pid="2" href="https://www.googl

In [31]:
base_url = "http://localhost:8003"  # 假设您的服务运行在本地8003端口

# 创建会话ID
session_id = "test_session_1"
url = f"{base_url}/workflow/tax_workflow/execute"
data = {
    "action_id": "navigate_to_main"
}
params = {
    "session_id": "test_session_1"
}

# 发送请求
response = requests.post(url, json=data, params=params)

In [32]:
response.json()

{'workflow_id': 'tax_workflow',
 'action_id': 'navigate_to_main',
 'result': {'status': 'success',
  'message': '成功打开纳税记录开具页面',
  'is_done': True,
  'task_success': True}}

In [3]:
#sys.path.append('/oper/work/endian/intelligent_agent')

In [165]:
user_api_key={"api-key": "chenhao"}#hao

In [33]:
user_api_key={"api-key": "wangendian"}#endian

In [38]:
response = requests.post(
    "http://localhost:8005/tabs",
    headers=user_api_key,
    json={"provider": "claude"}
)
# 获取tab_id用于后续操作
tab_id = response.json()#["tab_id"]
print(tab_id)

{'status': 'success', 'message': '已有claude标签页', 'tab_id': 'b67c4c88-8fd7-4eba-8150-aef07f16ae45', 'provider': 'claude', 'title': 'Automating Online Tasks with a Versatile AI Assistant - Claude', 'url': 'https://claude.ai/chat/bc3b6f4e-7a67-4879-add6-868a051365e8'}


In [196]:
response = requests.post(
    "http://localhost:8005/tabs/chatgpt/screenshot",
    headers=user_api_key,
    json={"provider": "chatgpt"}
)

In [197]:
response.json()

{'status': 'success',
 'screenshot_path': 'screenshots/screenshot_1743157130.png'}

In [45]:
def send_wechat_message():
    processed_params = {
        "contact_name": "陈浩",
        "message": "测试微信数据接口 "
    }
    response = requests.post(
        "http://localhost:8003/tools/wechat/search_and_send",
        json=processed_params,
        timeout=300.0
    )
    return response.json()

In [412]:
def check_tax():
    processed_params = {
        "city": "深圳",
        "start_date": "2025-01-01",
        "end_date": "2025-04-01"
    }
    response = requests.post(
        "http://localhost:8003/tools/tax/navigate_to_main",
        json=processed_params,
        timeout=300.0
    )
    return response.json()

In [436]:
response=check_tax()
print(response)

{'status': 'success', 'message': '成功打开纳税记录开具页面', 'tab_id': 2, 'url': 'https://its.shenzhen.chinatax.gov.cn:4433/gkpt/#/taxChecklist'}


In [424]:
def check_ssn():
    processed_params = {
    }
    response = requests.post(
        "http://localhost:8003/tools/social_security/navigate_and_select_person",
        json=processed_params,
        timeout=300.0
    )
    return response.json()

In [452]:
response=check_ssn()
print(response)

{'status': 'success', 'message': '社保清单查询完成并已下载', 'element_index': 2, 'element_text': '下载', 'new_tab_created': False, 'click_result': {'status': 'success', 'message': '成功点击元素: 下载', 'element_index': 2, 'element_text': '下载', 'new_tab_created': False, 'new_tab_ids': [], 'url': 'https://sipub.sz.gov.cn/hspms/logon.do?code=pm01_qQ5_fycIRsW9iV5UJgAeTg&gdbstoken=null&method=gdCasCallback&sxbm=btnDoPrintSbcbzm###', 'title': '深圳市社会保险基金管理局-个人网上服务', 'elements_count': 6}, 'is_done': True, 'task_success': True}


In [39]:
# 发送消息给Claude
response = requests.post(
    "http://localhost:8005/chat/claude",
    headers=user_api_key,
    json={
        "tab_id": tab_id,
        "prompt": "stop",
        "file_paths":None,
        "new_chat": False
    }
)
print(response.json())

{'id': 'chatcmpl-bc3b6f4e-7a67-4879-add6-868a051365e8', 'created': 1744806526, 'model': 'Claude 3.7 Sonnet', 'messages': [{'role': 'user', 'content': '# 通用 Agent 系统提示  你是一个通用 AI 助手，负责控制浏览器自动化系统来帮助用户完成各种在线任务。你能够理解用户的自然语言请求，并将其转化为一系列浏览器操作来完成任务。当遇到需要用户手动介入的情况（如登录、验证码、敏感信息输入等），你应该暂停自动化流程，请求用户手动操作，然后在用户确认完成后继续执行。  ## 工作流理解与匹配  当接收到用户请求时，你需要深入理解用户意图，并智能判断： 1. 用户请求的目标是什么 2. 如何高效执行任务  ## 工作流程  1. 理解用户的请求 2. 执行每个步骤并观察结果 3. 根据结果调整下一步行动 3. 在需要时请求用户交互 4. 向用户报告进度和最终结果  ## 用户交互指南  当遇到以下情况时，你应该暂停并请求用户手动介入：  1. 需要登录: 当页面需要用户名/密码登录时 2. 需要选择: 当有多个选项但无法自动确定正确选项时 3. 需要安全验证: 当出现验证码、短信验证或人脸识别等安全验证时 4. 敏感信息输入: 当需要输入身份证号、银行卡号等敏感信息时 5. 决策选择: 当需要用户做出重要决策时  ## 用户交互操作  在需要用户交互时，使用以下操作：  1. request_user_action: 请求用户手动执行操作     json    {      "request_user_action": {        "type": "login|select|verify|input|decision",        "message": "请登录您的账号",        "description": "系统需要您的账号密码，请手动完成登录",        "options": ["选项1", "选项2"]  // 可选，提供选择项      }    }      2. evaluate_state: 用户操作完成后评估当前状态     json    {      "evaluate_st

In [44]:
response.json()['messages'][1:-1]

[{'role': 'assistant',
  'content': {'response': ['好的，我将帮您打开小红书网站，提醒您登录，然后搜索中信银行信用卡并打开第一个结果。我会一步步执行这个任务并在需要您手动登录时通知您。'],
   'codeBlocks': [{'code': 'json{\n  "current_state": {\n    "evaluation_previous_goal": "理解用户的请求：用户需要我帮助打开小红书网站(xiaohongshu.com)，提醒用户登录，然后搜索\'中信银行信用卡\'并点击第一个搜索结果进入详情页面。",\n    "memory": "任务刚开始，尚未执行任何步骤。",\n    "next_goal": "打开小红书网站",\n    "user_interaction_needed": false\n  },\n  "action": [\n    {\n      "action_name": "create_tab",\n      "parameters": {\n        "url": "https://www.xiaohongshu.com"\n      }\n    }\n  ]\n}',
     'isInline': True,
     'language': 'json'}],
   'documents': [],
   'codeExplanations': []}},
 {'role': 'user',
  'content': '以下是您上次动作的结果:  [当前状态开始] 状态: ❌ 错误 消息: 执行动作失败: \'str\' object is not a mapping [当前状态结束]  基于此结果，下一步应该执行什么动作来完成用户的请求: "请帮我打开小红书xiaohongshu.co                                  m并提醒我登录，然后搜索中信银行信用卡并点击第一个返回的结果页面打开                                  进入页面"？  请记住： 1. 使用合理的操作顺序：对于搜索功能，应先点击搜索框，然后输入文本，最后点击搜索按钮 2. 使用highlight_elemen

In [158]:
# 发送消息给Claude
response = requests.post(
    "http://localhost:8005/chat/claude",
    headers=user_api_key,
    json={
        "tab_id": tab_id,
        "prompt": """写个执行python代码的命令
""",
        "file_paths":None,
        "new_chat": True
    }
)
#print(response.json())

In [155]:
print(response.json())

{'status': 'error', 'message': 'Could not start new chat'}


In [406]:
response = requests.post(
    "http://localhost:8005/chat/claude",
    headers=user_api_key,
    json={
        "tab_id": tab_id,
        "prompt": "stop",
        "file_paths":None,
        "new_chat": False
    }
)

KeyboardInterrupt: 

In [205]:
response.json()['messages'][-1]['content']['codeBlocks'][-1]

{'code': 'const rect = el.getBoundingClientRect();\n                        return text.length > 50 && // Has substantial text\n                              rect.width > 100 && // Has reasonable width\n                              !el.querySelector(\'pre\') && // Not just code\n                              rect.height > 30; // Not too small\n                    });\n                \n                // Group them into likely conversation turns by analyzing text patterns\n                const allText = textElements.map(el => el.textContent.trim());\n                \n                // Try to identify user vs assistant messages\n                const userPatterns = [\n                    /^you:/i, /^user:/i, /^human:/i, \n                    /^\\s*[A-Z]\\s+/, // Single letter prefix (like Claude\'s "H")\n                    /edit$/i, // Edit button suffix\n                ];\n                \n                const assistantPatterns = [\n                    /^claude:/i, /^assistant:

In [88]:
codeblock_list=[CodeBlock(language=line['language'],code=line['code']) for line in response.json()['messages'][-3]['content']['codeBlocks'] if line['language'] in ['python','bash','sh'] ]

In [93]:
code_exe_result=await LocalCodeExecutor(codeblock_list,env='/oper/work/endian/LLM-Assistant/py310/',filedir='/oper/work/endian/intelligent_agent/page_analyzer')

In [94]:
code_exe_result

"Added to path: /oper/work/endian/intelligent_agent\nSuccessfully imported BrowserSession\n正在查找包含 chatgpt.com 的标签页...\n找到包含 chatgpt.com 的标签页: FB8A15F3DA00BE968FBA2E302E72714A\n当前所有标签页: ['FB8A15F3DA00BE968FBA2E302E72714A', '6B8FB9E909F569A8CAB0476D21BBD064']\n目标标签页句柄: FB8A15F3DA00BE968FBA2E302E72714A\n"

In [37]:
#获取所有标签页：
response = requests.get(
    "http://localhost:8005/tabs",
    headers=user_api_key
)
tabs = response.json()
print(tabs)

[{'tab_id': '46e2a78c-b426-4cac-b736-66cf713beeae', 'provider': 'claude', 'title': 'Quantum CUBO Code Example - Claude', 'url': 'https://claude.ai/chat/02512880-b168-425b-a9d1-0b95b44f092b'}]


In [26]:
import uuid

def generate_api_key():
    """生成一个随机的API密钥"""
    return str(uuid.uuid4())

# 示例使用
new_user_key = generate_api_key()

In [30]:
generate_api_key()

'f16ab7cb-c2d9-4f2a-b2a9-968a0df75385'